# Benchmark JAX GPU Colab — PR #14f (Sprint 8 warmup+chunked + Sprint 9 pmap)

Novas APIs:
- `warmup_all_buckets(ctx)` — pré-compila todos os JITs **antes** do loop cronometrado.
- `forward_pure_jax_chunked(rho_h, rho_v, ctx, chunk_size=16)` — reduz pico de VRAM.
- `forward_pure_jax_pmap(rho_h_batch, rho_v_batch, ctx)` — distribui em N GPUs (A100 × 4).

In [ ]:
!pip install -q numpy 'numba>=0.60' 'jax[cuda12]>=0.4.30'
!git clone -b main https://github.com/daniel-guitarplayer-8/geosteering-ai.git /content/Geosteering_AI
%cd /content/Geosteering_AI

In [ ]:
import os, sys, time
os.environ['JAX_ENABLE_X64'] = 'True'
sys.path.insert(0, '/content/Geosteering_AI')

import jax
import numpy as np
jax.config.update('jax_enable_x64', True)
print('JAX devices:', jax.devices(), '| n_devices:', jax.local_device_count())
print('Default backend:', jax.default_backend())

## 1. Sprint 8 — warmup + forward pós-warmup (amortiza JIT)

In [ ]:
from geosteering_ai.simulation._jax.forward_pure import (
    build_static_context, forward_pure_jax,
    forward_pure_jax_chunked, warmup_all_buckets,
    clear_jit_cache, set_jit_cache_maxsize, get_jit_cache_info,
)
from geosteering_ai.simulation.validation.canonical_models import get_canonical_model

def bench_with_warmup(model_name, n_pos=100, n_iter=5):
    clear_jit_cache()
    m = get_canonical_model(model_name)
    z = np.linspace(m.min_depth - 2.0, m.max_depth + 2.0, n_pos)
    ctx = build_static_context(m.rho_h, m.rho_v, m.esp, z,
                                 np.array([20000.0]), 1.0, 0.0)
    t0 = time.perf_counter()
    n_buckets = warmup_all_buckets(ctx)
    t_warmup = time.perf_counter() - t0
    t0 = time.perf_counter()
    for _ in range(n_iter):
        H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx); H.block_until_ready()
    t_med = (time.perf_counter() - t0) / n_iter
    return t_warmup, t_med, n_buckets

print(f'{"Modelo":<20}{"warmup (s)":>12}{"forward (ms)":>15}{"buckets":>10}')
for name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28', 'hou_7', 'viking_graben_10']:
    t_w, t_f, nb = bench_with_warmup(name, n_pos=100, n_iter=3)
    print(f'{name:<20}{t_w:>12.1f}{t_f*1000:>14.2f}{nb:>10}')

## 2. Sprint 8 — chunked reduz pico de VRAM

In [ ]:
clear_jit_cache()
m = get_canonical_model('oklahoma_28')
z = np.linspace(m.min_depth - 2, m.max_depth + 2, 100)
ctx = build_static_context(m.rho_h, m.rho_v, m.esp, z,
                             np.array([20000.0]), 1.0, 0.0)

for chunk in (16, 32, 64, 100):
    t0 = time.perf_counter()
    H = forward_pure_jax_chunked(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx, chunk_size=chunk)
    H.block_until_ready()
    t = time.perf_counter() - t0
    try:
        peak = jax.devices()[0].memory_stats().get('peak_bytes_in_use', 0) / (1024**3)
    except Exception:
        peak = None
    print(f'chunk_size={chunk:>3}: {t*1000:.1f} ms | peak VRAM = {peak:.2f} GB' if peak else f'chunk_size={chunk:>3}: {t*1000:.1f} ms')

## 3. Sprint 9 — pmap multi-GPU (funciona em 1 GPU ou N GPUs)

In [ ]:
from geosteering_ai.simulation._jax.forward_pure import forward_pure_jax_pmap

n_devices = jax.local_device_count()
print(f'Número de devices: {n_devices}')

clear_jit_cache()
m = get_canonical_model('oklahoma_5')
z = np.linspace(m.min_depth - 2, m.max_depth + 2, 80)
ctx = build_static_context(m.rho_h, m.rho_v, m.esp, z,
                             np.array([20000.0]), 1.0, 0.0)

# Cria um batch com n_devices modelos (variando ρ para simular diferentes cenários)
rho_h_batch = jax.numpy.stack([ctx.rho_h_jnp * (1.0 + 0.1*k) for k in range(n_devices)])
rho_v_batch = jax.numpy.stack([ctx.rho_v_jnp * (1.0 + 0.1*k) for k in range(n_devices)])

# Warmup
H_pmap = forward_pure_jax_pmap(rho_h_batch, rho_v_batch, ctx)
H_pmap.block_until_ready()

t0 = time.perf_counter()
for _ in range(3):
    H_pmap = forward_pure_jax_pmap(rho_h_batch, rho_v_batch, ctx)
    H_pmap.block_until_ready()
t = (time.perf_counter() - t0) / 3
print(f'pmap {n_devices} modelos em paralelo: {t*1000:.2f} ms ({n_devices*3600/t:.0f} mod/h)')